# Modelling
## Data Preparation & Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting settings
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load Data
summary = pd.read_csv('./Ecommerce data/summary2018-19.csv')
survey = pd.read_csv('./Ecommerce data/survey.csv')
purchases = pd.read_csv('./Ecommerce data/purchases.csv')

# --- Feature Engineering ---
print(">>> Constructing Advanced Features...")

# 1. Demographics (from Survey)
# Clean whitespace
for col in survey.select_dtypes(include='object').columns:
    survey[col] = survey[col].str.strip()

# Binary Signals
survey['Is_Parent'] = survey['Q-life-changes'].apply(lambda x: 1 if isinstance(x, str) and ('Had a child' in x or 'Became pregnant' in x) else 0)
survey['Is_Mover'] = survey['Q-life-changes'].apply(lambda x: 1 if isinstance(x, str) and 'Moved place of residence' in x else 0)
high_income_cats = ['$100,000 - $149,999', '$150,000 or more']
survey['High_Income'] = survey['Q-demos-income'].apply(lambda x: 1 if x in high_income_cats else 0)

# 2. Behavioral (from Purchases)
purchases['Order Date'] = pd.to_datetime(purchases['Order Date'])

# Entropy
def calculate_entropy(x):
    return entropy(x.value_counts())

user_entropy = purchases.groupby('Survey ResponseID')['Category'].agg(calculate_entropy).reset_index()
user_entropy.columns = ['respondentid', 'Category_Entropy']

# Seasonality (Q4 Ratio)
purchases['Is_Q4'] = purchases['Order Date'].dt.month.isin([11, 12])
user_seasonal = purchases.groupby('Survey ResponseID').apply(
    lambda x: x[x['Is_Q4']]['Purchase Price Per Unit'].sum() / x['Purchase Price Per Unit'].sum() if x['Purchase Price Per Unit'].sum() > 0 else 0
).reset_index(name='Q4_Spend_Ratio')
user_seasonal.rename(columns={'Survey ResponseID': 'respondentid'}, inplace=True)

# 3. Merge
final_df = summary.merge(survey[['Survey ResponseID', 'Is_Parent', 'Is_Mover', 'High_Income']], 
                         left_on='respondentid', right_on='Survey ResponseID', how='left')
final_df = final_df.merge(user_entropy, on='respondentid', how='left')
final_df = final_df.merge(user_seasonal, on='respondentid', how='left')

# Fill NaNs
final_df[['Category_Entropy', 'Q4_Spend_Ratio']] = final_df[['Category_Entropy', 'Q4_Spend_Ratio']].fillna(0)

print(f"Data Prepared. Shape: {final_df.shape}")

## Preprocessing & Modeling

In [ ]:
# --- Preprocessing ---
print(">>> Preprocessing & Scaling...")

model_df = final_df.copy()

# 1. Log Transform Skewed Metrics (Add 1 to avoid log(0))
cols_to_log = ['totalspend', 'totalpurchases', 'avgdailyspend']
for col in cols_to_log:
    model_df[f'Log_{col}'] = np.log1p(model_df[col])

# 2. Select Features for Clustering
# We combine RFM (Financial) + Demographics (Profile) + Behavior (Entropy/Seasonality)
features = [
    'Log_totalspend', 'Log_totalpurchases', 'purchasedays',  # RFM
    'High_Income', 'Is_Parent', 'Is_Mover',                 # Life Stage / Profile
    'Category_Entropy', 'Q4_Spend_Ratio'                    # Behavior
]

# 3. Scaling
scaler = StandardScaler()
model_scaled = scaler.fit_transform(model_df[features])

# --- Modeling ---
print(">>> Running K-Means (K=5)...")
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
model_df['Cluster'] = kmeans.fit_predict(model_scaled)

# Calculate Cluster Sizes
print(model_df['Cluster'].value_counts().sort_index())

## Cluster Profiling(The "Persona" Reveal)

In [ ]:
# --- Cluster Interpretation ---
print(">>> Generating Cluster Profiles...")

# Group by Cluster and calculate mean of original (interpretable) features
# We map the cluster IDs to a readable summary table
profile_cols = [
    'totalspend', 'totalpurchases',  # Financial
    'High_Income', 'Is_Parent', 'Is_Mover', # Profile (Mean 0-1 represents %)
    'Category_Entropy', 'Q4_Spend_Ratio' # Behavior
]

cluster_profile = model_df.groupby('Cluster')[profile_cols].mean()

# Add Count column
cluster_profile['Count'] = model_df['Cluster'].value_counts()
cluster_profile['Percent'] = (cluster_profile['Count'] / len(model_df) * 100).round(1)

print("\n--- Cluster Summary Table ---")
print(cluster_profile.round(2))

# --- Visualization: Heatmap ---
# Normalize data for heatmap (Standardize columns to compare relative strength)
# (x - mean) / std
profile_normalized = (cluster_profile[profile_cols] - cluster_profile[profile_cols].mean()) / cluster_profile[profile_cols].std()

plt.figure(figsize=(14, 6))
sns.heatmap(profile_normalized.T, cmap='RdBu_r', center=0, annot=True, fmt='.2f')
plt.title('Cluster DNA: Relative Feature Strength (Red=High, Blue=Low)')
plt.xlabel('Cluster ID')
plt.show()

## Naming & Strategy (Automated Logic)

In [ ]:
# --- Automated Persona Tagging ---
def tag_cluster(row):
    tags = []
    
    # Financial Tags
    if row['totalspend'] > 3000: tags.append("Big Spender")
    elif row['totalspend'] < 500: tags.append("Low Value")
    
    # Profile Tags
    if row['Is_Parent'] > 0.3: tags.append("New Parent") # If >30% are parents (high concentration)
    if row['Is_Mover'] > 0.3: tags.append("Mover")
    if row['High_Income'] > 0.6: tags.append("Affluent")
    
    # Behavior Tags
    if row['Q4_Spend_Ratio'] > 0.4: tags.append("Seasonal")
    if row['Category_Entropy'] > 1.8: tags.append("Generalist")
    elif row['Category_Entropy'] < 0.5: tags.append("Specialist")
    
    return " / ".join(tags) if tags else "Standard User"

cluster_profile['Persona'] = cluster_profile.apply(tag_cluster, axis=1)

print("\n--- Final Personas & Strategy ---")
for idx, row in cluster_profile.iterrows():
    print(f"Cluster {idx} ({row['Percent']}%): [{row['Persona']}]")
    print(f"   > Avg Spend: ${row['totalspend']:.0f} | Is Parent: {row['Is_Parent']:.1%} | Is Mover: {row['Is_Mover']:.1%}")
    
    # Simple Strategy Logic
    strategy = ""
    if "New Parent" in row['Persona']: strategy = "Cross-sell Baby Products + Subscription"
    elif "Mover" in row['Persona']: strategy = "Win-back Campaign (Home/Furniture Discounts)"
    elif "Big Spender" in row['Persona']: strategy = "VIP Concierge (No Discounts)"
    elif "Seasonal" in row['Persona']: strategy = "Q4 Early Access / Pre-order"
    elif "Low Value" in row['Persona']: strategy = "Activation Coupon"
    else: strategy = "Standard Retention"
    
    print(f"   > Strategy: {strategy}\n")

### 📊 K-Means Clustering (V2.0) 决策逻辑与结果分析

这份文档总结了我们如何从原始的 RFM 模型进化到结合了**人口统计 (Demographics)** 和 **行为特征 (Behavior)** 的 V2.0 高级聚类模型。我们的目标是构建一个不仅能区分“有钱/没钱”，还能识别“用户是谁/在经历什么”的三维细分体系。

#### 1\. 特征工程决策 (Feature Engineering Decisions)

我们选择引入以下新特征，基于 EDA 中的核心发现：

  * **`Is_Parent` (新晋父母)**:
      * *逻辑:* EDA 显示“生子”事件带来 18% 的消费溢价。这是一个极强的高价值信号。
      * *目的:* 识别高潜力增长群体。
  * **`Is_Mover` (搬家者)**:
      * *逻辑:* EDA 显示搬家者消费下降 8%，且在家具品类上表现出明显的购买流失。
      * *目的:* 识别流失风险群体，进行针对性挽回。
  * **`Category_Entropy` (品类熵)**:
      * *逻辑:* 区分“只买猫砂的垂直用户”与“什么都买的忠诚用户”。
      * *目的:* 决定是推荐单一品类还是全站爆款。
  * **`Q4_Spend_Ratio` (季节性依赖)**:
      * *逻辑:* 识别那些只在黑五/圣诞大促期间活跃的“价格敏感型”用户。

#### 2\. 模型配置决策 (Model Configuration)

  * **算法选择**: K-Means。
      * *原因:* 依然选择 K-Means 而非 K-Prototypes，是因为我们将所有分类特征（如 `Is_Parent`）都转化为了二进制数值 (0/1)，且进行了 Standard Scaler 标准化。在这种特征空间下，欧氏距离依然有效且解释性最强。
  * **K值选择**: K=5。
      * *原因:* 之前的 K=4 主要基于 RFM 区分了 High/Low/Loyal/Risk。我们将 K 增加到 5，是为了给模型空间去捕捉新加入的特征（主要是 Parent 和 Mover 这两个稀疏但重要的信号），期望模型能将他们剥离为独立群体。

#### 3\. 聚类结果解读 (The "Personas" Reveal)

根据代码输出 (`31016_finalproj_model_indi.html`)，K-Means (K=5) 成功识别出了极其清晰的 5 个群体：

| Cluster ID | 占比 | 标签 (Persona) | 核心特征 (Key DNA) | 战略定位 |
| :--- | :--- | :--- | :--- | :--- |
| **Cluster 0** | **4.7%** | **High-Value New Parents** <br> (高价值新晋父母) | **Parents: 100%** <br> Spend: **$3,254** | **增长引擎** <br> 他们是全站最高价值潜力的群体。刚需强，价格敏感度低。 |
| **Cluster 3** | **15.2%** | **At-Risk Movers** <br> (流失风险搬家者) | **Movers: 100%** <br> Spend: $1,980 | **挽回重点** <br> 消费力中等，但正在经历生活变动，极易流失到竞争对手。 |
| **Cluster 4** | **19.2%** | **Affluent VIPs** <br> (富裕忠诚客) | Spend: **$7,028\*\* (Max) <br> Income: High | **现金牛** <br> 贡献了绝大部分利润。不需要折扣，需要服务和尊享感。 |
| **Cluster 1** | **37.6%** | **Standard Generalists** <br> (标准大众用户) | Spend: $1,857 <br> No specific life event | **基本盘** <br> 最大的群体。维持现状，定期大促激活即可。 |
| **Cluster 2** | **23.3%** | **Low-Value / Dormant** <br> (低价值/沉睡) | Spend: **$249\*\* (Min) <br> Activity: Low | **长尾** <br> 偶尔买点小东西。不用投入太多资源，用自动化 EDM 维护。 |

#### 4\. 关键结论 (Verdict)

V2.0 模型取得了巨大成功。

  * **解离成功:** 模型完美地将 **Parents (Cluster 0)** 和 **Movers (Cluster 3)** 从大众用户中剥离了出来。
  * **MECE 原则:** 群体之间互斥且穷尽，每个用户都有明确归属。
  * **可执行性:** 每个群体对应的营销策略（Strategy）清晰且各不相同（如 Cluster 0 推订阅，Cluster 3 推家具折扣）。

-----

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# --- 1. Load Data ---
# 请确保当前目录下有这些文件
summary = pd.read_csv('./Ecommerce data/summary2018-19.csv')
survey = pd.read_csv('./Ecommerce data/survey.csv')
purchases = pd.read_csv('./Ecommerce data/purchases.csv')
summary_future = pd.read_csv('./Ecommerce data/summary2020-21.csv')


# --- 2. Feature Engineering (特征工程) ---
# 清理文本空格
for col in survey.select_dtypes(include='object').columns:
    survey[col] = survey[col].str.strip()

# 构建二进制信号
survey['Is_Parent'] = survey['Q-life-changes'].apply(lambda x: 1 if isinstance(x, str) and ('Had a child' in x or 'Became pregnant' in x) else 0)
survey['Is_Mover'] = survey['Q-life-changes'].apply(lambda x: 1 if isinstance(x, str) and 'Moved place of residence' in x else 0)
high_income_cats = ['$100,000 - $149,999', '$150,000 or more']
survey['High_Income'] = survey['Q-demos-income'].apply(lambda x: 1 if x in high_income_cats else 0)

# 构建行为特征
purchases['Order Date'] = pd.to_datetime(purchases['Order Date'])

# 计算品类熵 (Category Entropy)
def calculate_entropy(x):
    return entropy(x.value_counts())

user_entropy = purchases.groupby('Survey ResponseID')['Category'].agg(calculate_entropy).reset_index()
user_entropy.columns = ['respondentid', 'Category_Entropy']

# 计算 Q4 消费占比 (Seasonality)
purchases['Is_Q4'] = purchases['Order Date'].dt.month.isin([11, 12])
user_seasonal = purchases.groupby('Survey ResponseID').apply(
    lambda x: x[x['Is_Q4']]['Purchase Price Per Unit'].sum() / x['Purchase Price Per Unit'].sum() if x['Purchase Price Per Unit'].sum() > 0 else 0
).reset_index(name='Q4_Spend_Ratio')
user_seasonal.rename(columns={'Survey ResponseID': 'respondentid'}, inplace=True)

# 合并特征表
final_df = summary.merge(survey[['Survey ResponseID', 'Is_Parent', 'Is_Mover', 'High_Income']], 
                         left_on='respondentid', right_on='Survey ResponseID', how='left')
final_df = final_df.merge(user_entropy, on='respondentid', how='left')
final_df = final_df.merge(user_seasonal, on='respondentid', how='left')
final_df[['Category_Entropy', 'Q4_Spend_Ratio']] = final_df[['Category_Entropy', 'Q4_Spend_Ratio']].fillna(0)

# --- 3. Modeling (K-Means V2) ---
model_df = final_df.copy()
# 对数转换处理长尾分布
cols_to_log = ['totalspend', 'totalpurchases', 'avgdailyspend']
for col in cols_to_log:
    model_df[f'Log_{col}'] = np.log1p(model_df[col])

features = [
    'Log_totalspend', 'Log_totalpurchases', 'purchasedays', 
    'High_Income', 'Is_Parent', 'Is_Mover',                 
    'Category_Entropy', 'Q4_Spend_Ratio'                    
]

# 标准化
scaler = StandardScaler()
model_scaled = scaler.fit_transform(model_df[features])

# 聚类 (K=5)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
model_df['Cluster'] = kmeans.fit_predict(model_scaled)

# --- 4. Dynamic Persona Mapping (动态标签映射) ---
# 自动根据特征均值给 Cluster 贴标签，防止 Cluster ID 随机变化
cluster_stats = model_df.groupby('Cluster')[['Is_Parent', 'Is_Mover', 'totalspend']].mean()

persona_map = {}
# 找到父母含量最高的群
parent_cluster = cluster_stats['Is_Parent'].idxmax()
persona_map[parent_cluster] = "High-Value New Parents"

# 找到搬家含量最高的群
mover_cluster = cluster_stats['Is_Mover'].idxmax()
persona_map[mover_cluster] = "At-Risk Movers"

# 剩下的里面，消费最高的叫 VIP
remaining = cluster_stats.drop([parent_cluster, mover_cluster])
vip_cluster = remaining['totalspend'].idxmax()
persona_map[vip_cluster] = "Affluent VIPs"

# 剩下的里面，消费最低的叫 Low Value
remaining = remaining.drop(vip_cluster)
low_cluster = remaining['totalspend'].idxmin()
persona_map[low_cluster] = "Low-Value / Dormant"

# 最后一个叫 Standard
remaining = remaining.drop(low_cluster)
std_cluster = remaining.index[0]
persona_map[std_cluster] = "Standard Generalists"

model_df['Persona'] = model_df['Cluster'].map(persona_map)

# --- 5. VALIDATION: Check Performance in 2020-21 (验证) ---
# 关联未来数据
validation_full = model_df[['respondentid', 'Persona', 'totalspend']].rename(columns={'totalspend': 'Spend_18_19'})
validation_full = validation_full.merge(
    summary_future[['respondentid', 'totalspend']].rename(columns={'totalspend': 'Spend_20_21'}),
    on='respondentid', 
    how='left'
)
# 填充空值为0 (代表在未来没有消费)
validation_full['Spend_20_21'] = validation_full['Spend_20_21'].fillna(0)
validation_full['Is_Active_20_21'] = validation_full['Spend_20_21'] > 0

# 计算验证指标
val_results = validation_full.groupby('Persona').agg({
    'respondentid': 'count',
    'Spend_18_19': 'mean',
    'Spend_20_21': 'mean',
    'Is_Active_20_21': 'mean' 
}).rename(columns={'respondentid': 'Count', 'Is_Active_20_21': 'Retention_Rate'})

val_results['Spend_Growth'] = (val_results['Spend_20_21'] - val_results['Spend_18_19']) / val_results['Spend_18_19']

print("--- VALIDATION: How did they perform in 2020-21? ---")
print(val_results.round(2).sort_values('Spend_20_21', ascending=False))

# --- 6. IMPACT SIZING (量化影响) ---
# 设定保守的提升假设
assumptions = {
    "High-Value New Parents": 0.10, # 增长很快，推订阅还能再涨10%
    "At-Risk Movers": 0.05,         # 挽回流失，提升5%
    "Affluent VIPs": 0.01,          # 主要是防守，提升1%
    "Standard Generalists": 0.02,   # 普调
    "Low-Value / Dormant": 0.05     # 基数低，激活容易，提升5%
}

impact_df = val_results.copy()
impact_df['Assumed_Lift'] = impact_df.index.map(assumptions)
# 计算基于未来预期的增量
impact_df['Projected_Revenue_Base'] = impact_df['Count'] * impact_df['Spend_20_21']
impact_df['Revenue_Uplift'] = impact_df['Projected_Revenue_Base'] * impact_df['Assumed_Lift']

print("\n--- IMPACT SIZING: Estimated Revenue Uplift ---")
print(impact_df[['Count', 'Spend_20_21', 'Assumed_Lift', 'Revenue_Uplift']].round(2).sort_values('Revenue_Uplift', ascending=False))

total_uplift = impact_df['Revenue_Uplift'].sum()
print(f"\n>>> TOTAL PROJECTED ANNUAL UPLIFT: ${total_uplift:,.0f}")

### 1\. 验证结果：谁在未来表现最好？(Validation Analysis)

我们用 2020-21 年的真实消费数据验证了 2018-19 年定义的群体。结果显示，所有群体的消费都在增长（这可能得益于 2020 年疫情期间电商的整体爆发），但增长的质量截然不同：

  * **High-Value New Parents (高价值新晋父母)**

      * **表现：** 消费从 $3,254 暴涨至 **$5,720\*\*，增长率 **+76%**。
      * **解读：** 假设完全成立。这群人正处于家庭消费的“黄金爬坡期”。他们的留存率是 100%，说明粘性极高。
      * **战略意义：** 这是最高质量的增长点。现在的投入（折扣/服务）在未来两年会带来 76% 的自然增值。

  * **At-Risk Movers (风险搬家者)**

      * **表现：** 消费从 $1,980 增长至 **$3,664\*\*，增长率 **+85%**。
      * **修正解读：** 之前我们担心他们会流失，但实际上他们留存很好（99%）且消费翻倍。这说明他们并没有“离开平台”，只是可能在“搬家特定品类”（如家具）上流失了，转而购买了更多其他商品。
      * **战略意义：** “挽回”策略应调整为\*\*“品类渗透”\*\*。他们还在买，只是没买家具。机会在于把家具卖给他们，而不是把人找回来。

  * **Low-Value / Dormant (低价值/沉睡)**

      * **表现：** 消费从 $249 激增至 **$1,159\*\*，增长率高达 **+366%**！
      * **惊喜发现：** 这群人并非“死粉”，而是“潜力股”。可能是之前未被激活，或者只是刚开始接触平台。
      * **战略意义：** 激活策略（Activation Coupon）的效果可能会比预期的好得多，因为他们本身就有巨大的自然增长动力。

  * **Affluent VIPs (富裕忠诚客)**

      * **表现：** 稳坐第一，人均消费 **$9,743**。
      * **解读：** 极其稳定（留存率 100%），是平台的压舱石。

-----

### 2\. 商业影响力预估 (Revenue Uplift Calculation)

根据你的代码输出，基于 2021 年的真实消费基数和保守的提升假设，**$528,520** 的增收机会分布如下：

| 排名 | 细分群体 (Persona) | 人数 | 2021人均消费 | 策略假设 | 预估增收 (Uplift) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **No.1** | **At-Risk Movers** | 712 | $3,664 | 5% (家具挽回) | **$130,441\*\* |
| **No.2** | **High-Value New Parents** | 218 | $5,720 | 10% (订阅/交叉) | **$124,702\*\* |
| **No.3** | **Standard Generalists** | 1,756 | $3,506 | 2% (普调) | **$123,138\*\* |
| **No.4** | **Affluent VIPs** | 895 | $9,743 | 1% (防守) | **$87,197\*\* |
| **No.5** | **Low-Value / Dormant** | 1,088 | $1,159 | 5% (激活) | **$63,042\*\* |

### 3\. 给 CTO 的最终洞察 (Final Verdict for CTO)

1.  **最大的机会在“搬家者” ($130k)**：

      * 不要被“流失风险”这个词误导。数据证明他们还在通过平台大量消费（增长 85%）。
      * **行动：** 不要发通用的“回来吧”邮件，要发精准的 **“新家硬装/软装”** 优惠券。他们有钱，也在花钱，只是没花在家具上。

2.  **最高效的投入是“新父母” ($125k)**：

      * 虽然只有 218 人，但贡献了全站第二高的增量。
      * **行动：** 针对这 200 多人建立**人工 VIP 客服通道**或**亲子订阅计划**。他们的 LTV (生命周期价值) 极高，值得重金投入。

3.  **不要忽视“沉睡用户”**：

      * 他们的自然增长率是最高的（3倍）。
      * **行动：** 继续维持低成本的自动化营销，他们正在自然转化为中等价值用户。

**结论：** 你的 V2.0 模型成功挖掘出了 **$52.8万** 的年度增收潜力，且验证了“搬家”和“生子”是比单纯的“消费金额”更具指向性的营销杠杆。